# ImgForge — metric improvement sweep (Kaggle)

One notebook, four stages, driven by `STAGE` in the first cell. Each stage runs a
short block so the session **always ends before Kaggle's 12 h cap** and the output
(including `persist/`) is committed:

| STAGE | What runs | GPU |
|---|---|---|
| `s0` | clone, dataset split, verification gate | no |
| `pilot` | EfficientNet-B0/ELA at 256 and 224 (20 epochs) + Swin/ELA sanity | yes |
| `sweep` | the 36 reported runs, one seed per session | yes |
| `analysis` | confound + McNemar over the persisted predictions | no |

`persist/` holds the log, checkpoints, predictions and the split CSVs. It is the
only thing that matters between sessions: save the version, create a **Dataset**
from `persist/`, and attach it to the next session. The ELA cache is ~14 GB and
stays in `/tmp` — it is recomputed once per session (~10 min) on purpose.

In [ ]:
STAGE = "s0"        # s0 -> pilot -> sweep -> analysis
SEEDS = "1"         # sweep stage only: "1", then "2", then "3" (one per session)
IMG = ""            # set to "--image-size 224" only if the 224 pilot wins

import glob, os, shutil, json

REPO, BRANCH = "https://github.com/UtkarshKumarJha/ImgForge.git", "metric-improvement"
WORK, PERSIST, CACHE = "/kaggle/working/ImgForge", "/kaggle/working/persist", "/tmp/ela_cache"

os.environ["GIT_TERMINAL_PROMPT"] = "0"
PERSIST = os.path.abspath(PERSIST)
print("stage:", STAGE or "all")

In [ ]:
!rm -rf {WORK}
!git clone -b {BRANCH} --depth 1 {REPO} {WORK}
%cd {WORK}
!git log --oneline -1
!pip install -q imagehash albumentations
!python -c "import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')"

## Restore everything the previous session persisted

Attach the dataset made from the previous version's `persist/` directory. With no
dataset attached this prints a warning and starts fresh — which is correct for the
first session only.

In [ ]:
os.makedirs(PERSIST, exist_ok=True)
candidates = sorted(set(glob.glob("/kaggle/input/*/persist") + glob.glob("/kaggle/input/*/*/persist")))
if candidates:
    src = candidates[0]
    print("restoring from", src)
    for sub in ("results", "models", "predictions", "data"):
        s, d = os.path.join(src, sub), os.path.join(PERSIST, sub)
        if not os.path.isdir(s):
            continue
        os.makedirs(d, exist_ok=True)
        n = sum(1 for f in os.listdir(s) if os.path.isfile(os.path.join(s, f)))
        for f in os.listdir(s):
            shutil.copy2(os.path.join(s, f), os.path.join(d, f))
        print(f"  {sub}: {n} files")
else:
    print("WARNING: no persist/ dataset attached — starting from scratch")

# persist/ is the durable copy; the CLIs read the working directories.
for sub in ("results", "models", "predictions", "data"):
    s = os.path.join(PERSIST, sub)
    if not os.path.isdir(s):
        continue
    os.makedirs(sub, exist_ok=True)
    for f in os.listdir(s):
        if os.path.isfile(os.path.join(s, f)):
            shutil.copy2(os.path.join(s, f), os.path.join(sub, f))
print("local:", {d: len(os.listdir(d)) for d in ("results", "models", "predictions", "data") if os.path.isdir(d)})

## S0 — dataset split + verification gate

Split generation is deterministic (sorted paths, fixed seed) but the perceptual-hash
dedup is slow, so it runs once and the CSVs are persisted. `verify_setup.py` exits 1
while the expected counts are still `PROVISIONAL_EXPECTED`; that is expected on the
first run — copy the printed fingerprint into the hand-off, not into the code by
guessing.

In [ ]:
CASIA = (glob.glob("/kaggle/input/*/CASIA2") + glob.glob("/kaggle/input/*/*/CASIA2"))
if not CASIA:
    raise SystemExit("CASIA2 not found under /kaggle/input — attach the dataset divg07/casia-20-image-tampering-detection-dataset")
RAW = "/kaggle/working/casia_link"
os.makedirs(RAW, exist_ok=True)
link = os.path.join(RAW, "CASIA2.0_revised")
if not os.path.islink(link):
    os.symlink(CASIA[0], link)
print("raw:", CASIA[0], "->", link)

if not (os.path.exists("data/train.csv") and os.path.exists("data/test.csv")):
    !python data/prepare_dataset.py --raw-dir {RAW} --out-dir data --seed 42
else:
    print("split CSVs restored from persist — not rebuilt")

os.makedirs(os.path.join(PERSIST, "data"), exist_ok=True)
for f in ("train.csv", "val.csv", "test.csv"):
    shutil.copy2(os.path.join("data", f), os.path.join(PERSIST, "data", f))

!wc -l data/train.csv data/val.csv data/test.csv
!python scripts/verify_setup.py --data-dir data --raw-dir {RAW} || true

## S1 — resolution pilot (20 epochs, EfficientNet-B0 / ELA)

Deviations are in the run name, so these never collide with the reported runs:
`_e20` for the short schedule, `_p224` for the 224 px arm. Compare validation
macro-F1, set `IMG` in the first cell, then move on.

In [ ]:
if STAGE == "pilot":
    !python run_sweep.py --archs efficientnet_b0 --input-modes ela --seeds 1 \
        --num-epochs 20 --persist-dir {PERSIST} --ela-cache-dir {CACHE} \
        --no-persist-ela-cache
    !python run_sweep.py --archs efficientnet_b0 --input-modes ela --seeds 1 \
        --num-epochs 20 --image-size 224 --persist-dir {PERSIST} --ela-cache-dir {CACHE} \
        --no-persist-ela-cache --skip-ela-warmup
    !python run_sweep.py --archs swin_tiny --input-modes ela --seeds 1 \
        --num-epochs 20 --persist-dir {PERSIST} --ela-cache-dir {CACHE} \
        --no-persist-ela-cache --skip-ela-warmup
else:
    print("skipped: stage is", STAGE)

In [ ]:
import pandas as pd
log_path = os.path.join(PERSIST, "experiment_log.csv")
if os.path.exists(log_path):
    log = pd.read_csv(log_path)
    pil = log[log.run_name.str.contains("_e20", na=False)]
    if len(pil):
        print(pil[["run_name", "f1_macro_tuned", "f1_macro_argmax",
                   "f1_macro_tuned_no_tta", "degenerate"]].to_string(index=False))
        best = pil.sort_values("f1_macro_tuned", ascending=False).iloc[0]
        print("\npilot winner:", best.run_name, "f1 =", best.f1_macro_tuned)
        print("degenerate:", pil[["run_name", "degenerate"]].to_string(index=False))
else:
    print("no log yet")

## S2 — the reported sweep

One seed per session (12 runs, ~8 h) so each session ends cleanly and commits.
Re-running the same command resumes: completed runs are skipped by name, and the
checkpoint, predictions and log were persisted after every run.

In [ ]:
if STAGE == "sweep":
    !python run_sweep.py --data-dir data --num-workers 2 --seeds {SEEDS} \
        --persist-dir {PERSIST} --ela-cache-dir {CACHE} --no-persist-ela-cache {IMG}
else:
    print("skipped: stage is", STAGE)

In [ ]:
import pandas as pd
log_path = os.path.join(PERSIST, "experiment_log.csv")
if os.path.exists(log_path):
    log = pd.read_csv(log_path)
    print(f"logged runs: {len(log)} / 36")
    if len(log):
        print(log[["run_name", "f1_macro_tuned", "f1_macro_argmax",
                   "f1_macro_tuned_no_tta", "degenerate"]]
              .sort_values("run_name").to_string(index=False))
    print("\npersist sizes:")
    for sub in ("results", "models", "predictions", "data"):
        d = os.path.join(PERSIST, sub)
        print(f"  {sub}: {len(os.listdir(d)) if os.path.isdir(d) else 0} files")
else:
    print("no log yet")

## S3 — analyses (needs all 36 runs)

Both CLIs read the threshold each run stored in its own `.npz`, so do not pass
`--threshold`. Localization needs CASIA ground-truth masks and may legitimately
report coverage ≈ 0; that is a documented outcome, not a failure.

In [ ]:
if STAGE == "analysis":
    !python -m analysis.confound --data-dir data --predictions predictions/*_test.npz \
        --out docs/results/confound.md
    !python -m analysis.mcnemar --predictions predictions/*_test.npz --models-dir models \
        --out docs/results/mcnemar.md
    !sed -n '1,60p' docs/results/mcnemar.md
else:
    print("skipped: stage is", STAGE)

## Ending a session

1. Check the status cell printed what you expect.
2. **Save Version → Quick Save** (the output includes `persist/`).
3. Create/refresh a **Dataset** from that version's `persist/`, and attach it to the
   next session. Without it the next session starts from zero.